# 03 — RAG Pipeline with Citations

This notebook covers milestone point **7**: a working Retrieval-Augmented Generation pipeline that answers maintenance questions using retrieved motor/VFD documentation and returns explicit sources.

In [1]:
import sys
from pathlib import Path
from dotenv import load_dotenv
import os

load_dotenv()
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from factory_floor.config import VECTOR_DIR, COLLECTION_NAME

if not os.getenv('OPENAI_API_KEY'):
    raise RuntimeError('Set OPENAI_API_KEY in a .env file before running this notebook.')


## 1. Load the existing vector database

No PDF ingestion or embedding should happen here. That work was already completed in notebook 02 and cached in Chroma.

In [2]:
from factory_floor.vectorstore import get_embeddings, load_vectorstore
from factory_floor.rag import build_retriever

embeddings = get_embeddings()
vectorstore = load_vectorstore(VECTOR_DIR, COLLECTION_NAME, embeddings=embeddings)
retriever = build_retriever(vectorstore, k=5)
print('Persistent retriever loaded.')


Persistent retriever loaded.


## 2. Define the maintenance RAG prompt

The model is explicitly told to use only retrieved context, admit when evidence is insufficient, and reference the source labels supplied in the context.

In [3]:
from factory_floor.rag import get_llm, PROMPT

llm = get_llm()
prompt = PROMPT


## 3. Build a deterministic citation layer

We label every retrieved chunk before it reaches the LLM and also append a source list programmatically. This makes the provenance visible even if the wording of the generated answer changes.

In [4]:
from factory_floor.rag import format_context, source_list


## 4. End-to-end RAG function

In [5]:
from factory_floor.rag import ask


def ask_factory_floor(question: str):
    return ask(question, retriever, llm)


## 5. Run the first end-to-end troubleshooting query

In [6]:
result = ask_factory_floor(
    'An industrial motor is overheating and has developed excessive vibration. What checks are supported by the manuals?'
)

print(result['answer'])
print('\nSOURCES USED')
print(result['sources'])

For an industrial motor experiencing overheating and excessive vibration, the manuals support the following checks:

1. Overheating Checks:
   - Verify ambient temperature is within defined limits.
   - Check load conditions and duty cycle configuration.
   - Confirm cooling system functionality.
   - Inspect motor load and reduce if necessary.
   - Check wiring and connection of the motor temperature sensor (KTY84 or PT1000).
   - Verify motor overtemperature parameters such as thermal time constant (p0611) and overtemperature fault threshold (p0605).
   - Check supply voltage parameterization (p0210) and line voltage.
   - Review torque limits (r1538, r1539) and current limits (p0640, r0067, r0289).
   - Perform motor data identification ensuring motor temperature is at ambient before starting.
   [SOURCE 1, SOURCE 3, SOURCE 5]

2. Vibration and Mechanical Checks:
   - Ensure all fixing bolts/screws for mechanical and electrical connections are securely tightened.
   - Confirm potent

## 6. Test a VFD-oriented question

In [7]:
result = ask_factory_floor(
    'A SINAMICS G120 drive is repeatedly tripping. According to the retrieved documentation, what information should a technician inspect before deciding on a cause?'
)
print(result['answer'])
print('\nSOURCES USED')
print(result['sources'])

Before deciding on a cause for repeated tripping of a SINAMICS G120 drive, a technician should inspect the following information from the fault and alarm documentation:

1. Check for specific fault codes and their causes:
   - Internal DRIVE-CLiQ communication faults: Verify DRIVE-CLiQ wiring and EMC-compliant installation [SOURCE 1].
   - Infeed faults: Inspect the line supply, filters, reactors, fuses, and infeed control [SOURCE 1].
   - Braking module faults: Check the braking module connection and temperature state [SOURCE 1].
   - DC link overvoltage faults: Review DC link voltage at trip time, motor regeneration, line supply voltage, phase interruptions, and DC link voltage controller settings (parameters p1121, p1130, p1136, p1240, p1280) [SOURCE 2].
   - DC link voltage minimum controller activation: Understand if the kinetic energy of the motor is buffering the DC link and if the power supply is stable [SOURCE 3].
   - Motor overcurrent faults: Verify motor current limits, cur

## 7. Test an out-of-corpus question

A good RAG system should not confidently answer questions that are unsupported by its corpus.

In [8]:
result = ask_factory_floor('How do I repair a hydraulic excavator boom cylinder?')
print(result['answer'])
print('\nSOURCES USED')
print(result['sources'])

The retrieved documentation does not provide specific instructions for repairing a hydraulic excavator boom cylinder. The sources mainly cover maintenance and repair procedures for Siemens electric motors (SIMOTICS series), including general repair guidelines, sealing measures, screw and bolt handling, and reassembly instructions [SOURCE 1, 2, 3, 4, 5].

For repairing a hydraulic excavator boom cylinder, you would typically need to:
- Disassemble the cylinder carefully, marking all fastening elements and internal connections for correct reassembly.
- Inspect and replace seals, check for damage to the cylinder barrel, piston, and rod.
- Apply appropriate sealants on sealing surfaces.
- Reassemble with correct torque on fasteners and ensure no damage to components.

However, since the retrieved context does not include detailed procedures or safety instructions specific to hydraulic excavator boom cylinders, the documentation is insufficient to provide a precise repair method.

Next, ver

## Milestone checkpoint

We now have:

`question → semantic retrieval → technical context → LLM → grounded answer + citations`

This is the core RAG that the Streamlit interface will expose.